# 📈 Stock Price Predictor Pro
## Advanced ML-Powered Stock Analysis & Prediction

**Features in this notebook:**
- 📊 Live stock data via `yfinance`
- 🔧 Rich technical indicators (RSI, MACD, Bollinger Bands, Moving Averages)
- 🤖 LSTM deep learning model for 7-day price forecasting
- 📉 Random Forest + Linear Regression ensemble comparison
- 📈 Interactive Plotly candlestick & signal charts
- 🟢🔴 Buy / Sell signal generation
- 📋 Model performance metrics dashboard


In [ ]:
# ── Install Dependencies ──────────────────────────────────────────────────────
!pip install yfinance plotly tensorflow scikit-learn pandas numpy matplotlib -q

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print('✅ All packages loaded successfully!')
print(f'TensorFlow version: {tf.__version__}')

## 1. 📥 Fetch Live Stock Data
We download 3 years of daily OHLCV data from Yahoo Finance.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
TICKER   = 'AAPL'    # Change to any ticker: 'TSLA', 'GOOGL', 'MSFT', etc.
PERIOD   = '3y'      # Data period
INTERVAL = '1d'      # Daily data
FORECAST_DAYS = 7    # Number of future days to predict

# ── Download ──────────────────────────────────────────────────────────────────
print(f'📥 Downloading {TICKER} data...')
raw = yf.download(TICKER, period=PERIOD, interval=INTERVAL, auto_adjust=True)

# Flatten MultiIndex columns if present
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

df = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
df.dropna(inplace=True)

print(f'✅ {len(df)} trading days loaded ({df.index[0].date()} → {df.index[-1].date()})')
df.tail()

## 2. 🔧 Feature Engineering — Technical Indicators
We compute a rich set of indicators used by professional traders.

In [ ]:
def add_technical_indicators(df):
    """Add a comprehensive set of technical indicators to the dataframe."""

    # ── Moving Averages ───────────────────────────────────────────────────────
    df['SMA_20']  = df['Close'].rolling(20).mean()
    df['SMA_50']  = df['Close'].rolling(50).mean()
    df['SMA_200'] = df['Close'].rolling(200).mean()
    df['EMA_12']  = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA_26']  = df['Close'].ewm(span=26, adjust=False).mean()

    # ── Bollinger Bands ───────────────────────────────────────────────────────
    rolling_std = df['Close'].rolling(20).std()
    df['BB_Upper'] = df['SMA_20'] + 2 * rolling_std
    df['BB_Lower'] = df['SMA_20'] - 2 * rolling_std
    df['BB_Width'] = df['BB_Upper'] - df['BB_Lower']

    # ── MACD ──────────────────────────────────────────────────────────────────
    df['MACD']        = df['EMA_12'] - df['EMA_26']
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist']   = df['MACD'] - df['MACD_Signal']

    # ── RSI (14-period) ───────────────────────────────────────────────────────
    delta = df['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))

    # ── Average True Range (ATR) ──────────────────────────────────────────────
    high_low  = df['High'] - df['Low']
    high_prev = (df['High'] - df['Close'].shift()).abs()
    low_prev  = (df['Low']  - df['Close'].shift()).abs()
    df['ATR'] = pd.concat([high_low, high_prev, low_prev], axis=1).max(axis=1).rolling(14).mean()

    # ── Price Momentum & Volatility ───────────────────────────────────────────
    df['Return_1d']  = df['Close'].pct_change(1)
    df['Return_5d']  = df['Close'].pct_change(5)
    df['Volatility'] = df['Return_1d'].rolling(20).std()

    # ── Volume Indicators ─────────────────────────────────────────────────────
    df['Volume_SMA'] = df['Volume'].rolling(20).mean()
    df['Volume_Ratio'] = df['Volume'] / df['Volume_SMA']

    # ── Target: Next Day Close ────────────────────────────────────────────────
    df['Target'] = df['Close'].shift(-1)

    return df


df = add_technical_indicators(df)
df.dropna(inplace=True)

print(f'✅ Technical indicators computed. Dataset shape: {df.shape}')
print(f'Features available: {list(df.columns)}')

## 3. 🟢🔴 Buy / Sell Signal Generation
Signals are based on RSI + MACD crossover + Bollinger Band rules.

In [ ]:
def generate_signals(df):
    """Generate trading signals using multiple technical indicator rules."""
    df = df.copy()

    # RSI signals
    rsi_buy  = df['RSI'] < 35
    rsi_sell = df['RSI'] > 65

    # MACD crossover signals
    macd_cross_up   = (df['MACD'] > df['MACD_Signal']) & (df['MACD'].shift(1) <= df['MACD_Signal'].shift(1))
    macd_cross_down = (df['MACD'] < df['MACD_Signal']) & (df['MACD'].shift(1) >= df['MACD_Signal'].shift(1))

    # Bollinger Band breakout
    bb_buy  = df['Close'] < df['BB_Lower']
    bb_sell = df['Close'] > df['BB_Upper']

    # Combined signal (at least 2 of 3 conditions agree)
    buy_score  = rsi_buy.astype(int)  + macd_cross_up.astype(int)  + bb_buy.astype(int)
    sell_score = rsi_sell.astype(int) + macd_cross_down.astype(int) + bb_sell.astype(int)

    df['Signal'] = 0  # Hold
    df.loc[buy_score  >= 2, 'Signal'] =  1   # Buy
    df.loc[sell_score >= 2, 'Signal'] = -1   # Sell

    return df


df = generate_signals(df)

buy_count  = (df['Signal'] ==  1).sum()
sell_count = (df['Signal'] == -1).sum()
hold_count = (df['Signal'] ==  0).sum()

print(f'🟢 Buy signals:  {buy_count}')
print(f'🔴 Sell signals: {sell_count}')
print(f'⚪ Hold signals: {hold_count}')

## 4. 📊 Interactive Plotly Charts

In [ ]:
def plot_price_with_signals(df, ticker):
    """Candlestick chart with moving averages, Bollinger Bands, and buy/sell signals."""

    # Use last 180 trading days for clarity
    plot_df = df.tail(180).copy()

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        row_heights=[0.6, 0.2, 0.2],
        vertical_spacing=0.04,
        subplot_titles=(f'{ticker} Price + Signals', 'MACD', 'RSI')
    )

    # ── Row 1: Candlestick + Indicators ──────────────────────────────────────
    fig.add_trace(go.Candlestick(
        x=plot_df.index,
        open=plot_df['Open'], high=plot_df['High'],
        low=plot_df['Low'],   close=plot_df['Close'],
        name='OHLC', increasing_line_color='#26a69a', decreasing_line_color='#ef5350'
    ), row=1, col=1)

    # Bollinger Bands
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['BB_Upper'], name='BB Upper',
                             line=dict(color='rgba(100,100,255,0.4)', dash='dot')), row=1, col=1)
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['BB_Lower'], name='BB Lower',
                             line=dict(color='rgba(100,100,255,0.4)', dash='dot'),
                             fill='tonexty', fillcolor='rgba(100,100,255,0.05)'), row=1, col=1)

    # Moving Averages
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['SMA_20'],  name='SMA 20',
                             line=dict(color='orange', width=1.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['SMA_50'],  name='SMA 50',
                             line=dict(color='purple', width=1.5)), row=1, col=1)

    # Buy / Sell signals
    buy_signals  = plot_df[plot_df['Signal'] ==  1]
    sell_signals = plot_df[plot_df['Signal'] == -1]

    fig.add_trace(go.Scatter(x=buy_signals.index,  y=buy_signals['Low']  * 0.98,
                             mode='markers', name='Buy Signal',
                             marker=dict(symbol='triangle-up', size=12, color='#00e676')), row=1, col=1)
    fig.add_trace(go.Scatter(x=sell_signals.index, y=sell_signals['High'] * 1.02,
                             mode='markers', name='Sell Signal',
                             marker=dict(symbol='triangle-down', size=12, color='#ff1744')), row=1, col=1)

    # ── Row 2: MACD ───────────────────────────────────────────────────────────
    colors = ['#26a69a' if v >= 0 else '#ef5350' for v in plot_df['MACD_Hist']]
    fig.add_trace(go.Bar(x=plot_df.index, y=plot_df['MACD_Hist'],
                         name='MACD Hist', marker_color=colors), row=2, col=1)
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['MACD'],
                             name='MACD', line=dict(color='#2196F3', width=1.5)), row=2, col=1)
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['MACD_Signal'],
                             name='Signal Line', line=dict(color='#FF9800', width=1.5)), row=2, col=1)

    # ── Row 3: RSI ────────────────────────────────────────────────────────────
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['RSI'],
                             name='RSI', line=dict(color='#9c27b0', width=2)), row=3, col=1)
    fig.add_hline(y=70, line_dash='dash', line_color='red',   row=3, col=1)
    fig.add_hline(y=30, line_dash='dash', line_color='green', row=3, col=1)

    fig.update_layout(
        title=f'{ticker} — Technical Analysis Dashboard',
        template='plotly_dark',
        height=900,
        showlegend=True,
        xaxis_rangeslider_visible=False
    )
    fig.show()


plot_price_with_signals(df, TICKER)

## 5. 🤖 ML Models — Feature Preparation

In [ ]:
# ── Feature columns for ML ────────────────────────────────────────────────────
FEATURE_COLS = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26',
    'BB_Upper', 'BB_Lower', 'BB_Width',
    'MACD', 'MACD_Signal', 'MACD_Hist',
    'RSI', 'ATR',
    'Return_1d', 'Return_5d', 'Volatility',
    'Volume_Ratio'
]

X = df[FEATURE_COLS].values
y = df['Target'].values

# 80/20 chronological split (NO shuffle — time-series)
split_idx = int(len(X) * 0.80)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'Train samples: {len(X_train)}   Test samples: {len(X_test)}')
print(f'Feature count: {X.shape[1]}')

## 6. 📉 Baseline Models — Linear Regression & Random Forest

In [ ]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'{name:<25}  MAE: ${mae:.2f}   RMSE: ${rmse:.2f}   R²: {r2:.4f}')
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}


# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

# Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print('─' * 60)
print('MODEL COMPARISON')
print('─' * 60)
lr_metrics = evaluate('Linear Regression',   y_test, lr_preds)
rf_metrics = evaluate('Random Forest (200)', y_test, rf_preds)
print('─' * 60)

## 7. 🧠 LSTM Deep Learning Model
LSTM (Long Short-Term Memory) networks are specifically designed for sequential/time-series data. We use a 60-day lookback window to capture temporal patterns.

In [ ]:
LOOKBACK = 60   # Days of history used as input sequence

# ── Scale data (LSTM is sensitive to scale) ───────────────────────────────────
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# Fit scaler ONLY on training data (avoid data leakage)
X_all_scaled = scaler_X.fit_transform(X)
y_all_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

# ── Build sequence windows ────────────────────────────────────────────────────
def make_sequences(X_scaled, y_scaled, lookback):
    Xs, ys = [], []
    for i in range(lookback, len(X_scaled)):
        Xs.append(X_scaled[i - lookback:i])
        ys.append(y_scaled[i])
    return np.array(Xs), np.array(ys)


X_seq, y_seq = make_sequences(X_all_scaled, y_all_scaled, LOOKBACK)

# Chronological split
split_seq = int(len(X_seq) * 0.80)
X_tr, X_ts = X_seq[:split_seq], X_seq[split_seq:]
y_tr, y_ts = y_seq[:split_seq], y_seq[split_seq:]

print(f'LSTM train shape: {X_tr.shape}   LSTM test shape: {X_ts.shape}')

In [ ]:
# ── Build LSTM Architecture ───────────────────────────────────────────────────
def build_lstm(input_shape):
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='huber')
    return model


lstm_model = build_lstm((LOOKBACK, len(FEATURE_COLS)))
lstm_model.summary()

In [ ]:
# ── Train LSTM ────────────────────────────────────────────────────────────────
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = lstm_model.fit(
    X_tr, y_tr,
    epochs=60,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

print(f'\n✅ Training complete — best val_loss: {min(history.history["val_loss"]):.6f}')

In [ ]:
# ── LSTM Predictions & Evaluation ────────────────────────────────────────────
lstm_preds_scaled = lstm_model.predict(X_ts).flatten()
lstm_preds = scaler_y.inverse_transform(lstm_preds_scaled.reshape(-1, 1)).flatten()
lstm_actual = scaler_y.inverse_transform(y_ts.reshape(-1, 1)).flatten()

lstm_metrics = evaluate('LSTM (3-layer)', lstm_actual, lstm_preds)
print('─' * 60)

# Training loss plot
fig_loss = go.Figure()
fig_loss.add_trace(go.Scatter(y=history.history['loss'],     name='Train Loss', line=dict(color='#2196F3')))
fig_loss.add_trace(go.Scatter(y=history.history['val_loss'], name='Val Loss',   line=dict(color='#FF5722')))
fig_loss.update_layout(title='LSTM Training Loss', xaxis_title='Epoch',
                       yaxis_title='Huber Loss', template='plotly_dark')
fig_loss.show()

## 8. 📊 Model Comparison — Actual vs Predicted

In [ ]:
# Align predictions (LSTM has LOOKBACK offset)
test_dates = df.index[split_idx:].tolist()

# For RF/LR the test set starts at split_idx
rf_test_dates   = test_dates
lstm_test_dates = df.index[split_idx + LOOKBACK:].tolist()

fig_cmp = go.Figure()
fig_cmp.add_trace(go.Scatter(x=rf_test_dates, y=y_test,      mode='lines',
                             name='Actual Close', line=dict(color='white',   width=2)))
fig_cmp.add_trace(go.Scatter(x=rf_test_dates, y=rf_preds,    mode='lines',
                             name='Random Forest', line=dict(color='#FF9800', width=1.5)))
fig_cmp.add_trace(go.Scatter(x=rf_test_dates, y=lr_preds,    mode='lines',
                             name='Linear Reg',   line=dict(color='#2196F3', width=1.5)))
fig_cmp.add_trace(go.Scatter(x=lstm_test_dates, y=lstm_preds, mode='lines',
                             name='LSTM',          line=dict(color='#00e676', width=2)))

fig_cmp.update_layout(
    title=f'{TICKER} — Model Predictions vs Actual Prices',
    xaxis_title='Date', yaxis_title='Price (USD)',
    template='plotly_dark', height=500
)
fig_cmp.show()

## 9. 🔮 7-Day Future Price Forecast

In [ ]:
def forecast_next_n_days(model, last_sequence, scaler_X, scaler_y, n_days=7):
    """
    Iterative multi-step forecast.
    Each day we predict the next close, then append it to the rolling window.
    """
    preds = []
    seq   = last_sequence.copy()  # shape: (LOOKBACK, features)

    for _ in range(n_days):
        inp   = seq[-LOOKBACK:].reshape(1, LOOKBACK, -1)
        pred_scaled = model.predict(inp, verbose=0)[0, 0]
        pred_price  = scaler_y.inverse_transform([[pred_scaled]])[0, 0]
        preds.append(pred_price)

        # Build next row: use last row and update Close with predicted price
        next_row = seq[-1].copy()
        close_idx = FEATURE_COLS.index('Close')
        pred_scaled_close = pred_scaled   # approximate: use pred as next scaled Close
        next_row[close_idx] = pred_scaled_close
        seq = np.vstack([seq, next_row])

    return preds


# Last sequence of scaled data
last_seq = X_all_scaled[-LOOKBACK:]
future_prices = forecast_next_n_days(lstm_model, last_seq, scaler_X, scaler_y, FORECAST_DAYS)

# Date range for future predictions (skip weekends)
last_date = df.index[-1]
future_dates = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=FORECAST_DAYS)

# ── Plot Forecast ─────────────────────────────────────────────────────────────
hist_tail = df['Close'].tail(60)

fig_fc = go.Figure()
fig_fc.add_trace(go.Scatter(x=hist_tail.index, y=hist_tail.values,
                            mode='lines', name='Historical Close',
                            line=dict(color='#90caf9', width=2)))
fig_fc.add_trace(go.Scatter(x=future_dates, y=future_prices,
                            mode='lines+markers', name='LSTM Forecast',
                            line=dict(color='#ffd54f', width=2, dash='dash'),
                            marker=dict(size=8, color='#ffd54f')))
# Confidence band (± 2% illustrative)
upper = [p * 1.02 for p in future_prices]
lower = [p * 0.98 for p in future_prices]
fig_fc.add_trace(go.Scatter(
    x=list(future_dates) + list(future_dates[::-1]),
    y=upper + lower[::-1],
    fill='toself', fillcolor='rgba(255,213,79,0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='±2% Confidence'
))

fig_fc.update_layout(
    title=f'{TICKER} — {FORECAST_DAYS}-Day LSTM Price Forecast',
    xaxis_title='Date', yaxis_title='Price (USD)',
    template='plotly_dark', height=450
)
fig_fc.show()

print('\n📅 Forecasted Prices:')
for d, p in zip(future_dates, future_prices):
    print(f'  {d.date()}  →  ${p:.2f}')

## 10. 🏆 Final Performance Dashboard

In [ ]:
# ── Summary Bar Chart ─────────────────────────────────────────────────────────
models = ['Linear Regression', 'Random Forest', 'LSTM']
all_preds = [lr_preds, rf_preds, lstm_preds]
all_actuals = [y_test, y_test, lstm_actual]

metrics_df = pd.DataFrame([
    {
        'Model': m,
        'MAE':  mean_absolute_error(a, p),
        'RMSE': np.sqrt(mean_squared_error(a, p)),
        'R²':   r2_score(a, p)
    }
    for m, a, p in zip(models, all_actuals, all_preds)
])

print('\n═══════════════ MODEL PERFORMANCE SUMMARY ═══════════════')
print(metrics_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

fig_bar = make_subplots(rows=1, cols=3,
                        subplot_titles=('Mean Absolute Error (lower=better)',
                                        'RMSE (lower=better)',
                                        'R² Score (higher=better)'))

colors_bar = ['#ef5350', '#FF9800', '#00e676']
for col, metric in enumerate(['MAE', 'RMSE', 'R²'], start=1):
    fig_bar.add_trace(
        go.Bar(x=metrics_df['Model'], y=metrics_df[metric],
               marker_color=colors_bar, name=metric,
               text=[f'{v:.2f}' for v in metrics_df[metric]],
               textposition='outside'),
        row=1, col=col
    )

fig_bar.update_layout(
    title='Model Performance Comparison',
    template='plotly_dark', height=400,
    showlegend=False
)
fig_bar.show()

In [ ]:
# ── Feature Importance (Random Forest) ────────────────────────────────────────
importances = rf.feature_importances_
feat_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=True).tail(15)

fig_imp = go.Figure(go.Bar(
    x=feat_df['Importance'], y=feat_df['Feature'],
    orientation='h',
    marker=dict(color=feat_df['Importance'], colorscale='Viridis')
))
fig_imp.update_layout(
    title='Random Forest — Top 15 Feature Importances',
    template='plotly_dark', height=500
)
fig_imp.show()

---
## 📝 Key Insights

| Aspect | Details |
|--------|----------|
| **Data** | 3 years daily OHLCV from Yahoo Finance |
| **Features** | 21 technical indicators (SMA, EMA, BB, MACD, RSI, ATR, momentum, volatility) |
| **LSTM Architecture** | 3-layer stacked LSTM (128→64→32) with Dropout regularization |
| **Lookback Window** | 60 trading days (~3 months) |
| **Forecast Horizon** | 7 trading days ahead |
| **Buy/Sell Signals** | Multi-indicator consensus (RSI + MACD crossover + Bollinger Band breakout) |

> **⚠️ Disclaimer**: This is for educational purposes only. Real trading requires much more rigorous backtesting, risk management, and should not be used as financial advice.